# Lung Sound Classification with Frozen Deep Learning Models
**IEEE BSN Workshop · Hands-on notebook**

<a href="https://colab.research.google.com/github/AlirezaFBabaei/BSN-Workshop/blob/main/BSN_Lung_Sounds.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In this notebook you go through the full workflow we use to classify respiratory cycles from the
[ICBHI 2017 Respiratory Sound Database](https://bhichallenge.med.auth.gr/) into four classes:
**Normal**, **Crackle**, **Wheeze** and **Both** (crackle and wheeze).

| Step | What you do |
|---|---|
| 1. Setup | Install and download the workshop data (about 1 minute) |
| 2. Listen to the data | Play real recordings from four different stethoscopes and see where the doctors marked abnormal sounds |
| 3. Filtering | Turn filters on and off and hear and see what they remove |
| 4. Segmentation | Cut recordings into breathing cycles of a fixed length |
| 5. Mel spectrogram | Turn a cycle into the image our vision models see |
| 6. Train a classifier | Pick a pretrained model and train a small classification head on its features |
| 7. Results | Measure performance with the official ICBHI metrics and look at predictions |
| 8. Export | Save your trained head as an ONNX file for the interactive website |

**How to use it:** run the cells from top to bottom (`Shift + Enter`). Cells with sliders and menus update
automatically when you change a value. Cells marked ✏️ are empty on purpose so you can try your own code.

## 1. Setup

This cell installs the two small packages that Google Colab does not include (`onnx`, `onnxruntime`),
imports everything, and downloads the workshop data (about 80 MB) from Hugging Face.
You only need to run it once.

In [ ]:
import importlib, subprocess, sys
for pkg in ["onnx", "onnxruntime", "librosa", "ipywidgets", "huggingface_hub"]:
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", pkg], check=True)

import json, os, warnings
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import librosa
import scipy.signal as sps
import torch
import torch.nn as nn
import ipywidgets as W
from IPython.display import Audio, display, clear_output
from sklearn.metrics import confusion_matrix, f1_score, accuracy_score

warnings.filterwarnings("ignore", category=UserWarning)
plt.rcParams.update({"figure.dpi": 100, "axes.grid": True, "grid.alpha": 0.3})

# Where the data comes from. Set BSN_DATA_DIR to use a local copy instead of downloading.
HF_REPO = "AlirezaFB/ICBHI_2017"
if os.environ.get("BSN_DATA_DIR"):
    DATA = Path(os.environ["BSN_DATA_DIR"])
else:
    from huggingface_hub import snapshot_download
    DATA = Path(snapshot_download(HF_REPO, repo_type="dataset", allow_patterns=["workshop/*"])) / "workshop"

MANIFEST = json.loads((DATA / "manifest.json").read_text())
CLASSES = [MANIFEST["label_map"][str(i)] for i in range(len(MANIFEST["label_map"]))]
CLASS_COLORS = {"Normal": "#9e9e9e", "Crackle": "#e67e22", "Wheeze": "#2e86de", "Both": "#c0392b"}
RECORDINGS = [r["recording"] for r in MANIFEST["sample_recordings"]["recordings"]]
print(f"Data folder: {DATA}")
print(f"Classes (in label order): {CLASSES}")
print(f"{len(RECORDINGS)} sample recordings, {len(MANIFEST['models'])} models")

### Helper functions
These small functions are used throughout the notebook. You don't need to change them,
but feel free to read them: they are the core of the preprocessing pipeline.

In [ ]:
def load_recording(name, sr=None):
    """Load a recording at its original sample rate (sr=None) or resampled to `sr` Hz."""
    audio, rate = librosa.load(DATA / "audio" / f"{name}.wav", sr=sr)
    return audio, rate


def load_cycles(name):
    """Read the doctors' annotations: one row per breathing cycle (start, end, label)."""
    rows = np.loadtxt(DATA / "audio" / f"{name}.txt", ndmin=2)
    names = {(0, 0): "Normal", (1, 0): "Crackle", (0, 1): "Wheeze", (1, 1): "Both"}
    return pd.DataFrame({"start": rows[:, 0], "end": rows[:, 1],
                         "label": [names[(int(c), int(w))] for c, w in rows[:, 2:4]]})


def butterworth(audio, sr, kind="band-pass", low=80.0, high=1900.0, order=5):
    """Apply a Butterworth filter. kind: 'none', 'low-pass', 'high-pass' or 'band-pass'."""
    nyq = sr / 2
    low, high = min(low, nyq * 0.99), min(high, nyq * 0.99)
    if kind == "none":
        return audio
    if kind == "low-pass":
        sos = sps.butter(order, high, btype="lowpass", fs=sr, output="sos")
    elif kind == "high-pass":
        sos = sps.butter(order, low, btype="highpass", fs=sr, output="sos")
    else:
        sos = sps.butter(order, [low, min(max(high, low + 1), nyq * 0.99)], btype="bandpass", fs=sr, output="sos")
    return sps.sosfilt(sos, audio)


def fix_length(cycle, n_samples, mode="reflect"):
    """Pad (at the end) or truncate a cycle to exactly n_samples."""
    if len(cycle) >= n_samples:
        return cycle[:n_samples]
    if mode == "zeros":
        return np.pad(cycle, (0, n_samples - len(cycle)))
    if mode == "repeat":
        return np.resize(cycle, n_samples)
    # reflect (mirror) padding; np.pad can only mirror up to len-1 samples per call, so repeat if needed
    out = cycle
    while len(out) < n_samples:
        out = np.pad(out, (0, min(n_samples - len(out), len(out) - 1)), mode="reflect")
    return out


def mel_db(audio, sr, n_fft=512, hop=128, n_mels=128, fmax=2000):
    """Mel spectrogram in decibels (relative to the loudest bin)."""
    mel = librosa.feature.melspectrogram(y=audio, sr=sr, n_fft=n_fft, hop_length=hop, n_mels=n_mels, fmax=fmax)
    return librosa.power_to_db(mel, ref=np.max)


def shade_cycles(ax, cycles, show_normal):
    """Colour each annotated cycle on a time axis."""
    for _, c in cycles.iterrows():
        if c.label == "Normal" and not show_normal:
            continue
        ax.axvspan(c.start, c.end, color=CLASS_COLORS[c.label], alpha=0.25, lw=0)
        ax.text((c.start + c.end) / 2, 0.97, c.label, transform=ax.get_xaxis_transform(),
                ha="center", va="top", fontsize=8, color=CLASS_COLORS[c.label], fontweight="bold")


def recording_label(r):
    return f"{r['recording']}  ({r['device']}, {r['original_sample_rate_hz']} Hz)"


def play(audio, sr, max_sr=8000):
    """Audio player. Lung sounds sit below 2 kHz, so we play at most 8 kHz to keep the notebook light."""
    if sr > max_sr:
        audio, sr = librosa.resample(np.asarray(audio, dtype=np.float32), orig_sr=sr, target_sr=max_sr), max_sr
    display(Audio(audio, rate=sr))


REC_OPTIONS = [(recording_label(r), r["recording"]) for r in MANIFEST["sample_recordings"]["recordings"]]

## 2. Listen to the data

ICBHI recordings come from **four different stethoscopes** (devices), recorded at different sample rates
and in different clinical settings. We picked 12 recordings, 3 per device, each containing at least one
abnormal cycle.

Each recording comes with an annotation file written by clinicians: for every breathing cycle it gives
the start time, end time, and whether a **crackle** and/or a **wheeze** was heard.

In [ ]:
table = pd.DataFrame(MANIFEST["sample_recordings"]["recordings"])
table = pd.concat([table.drop(columns="cycles_per_class"),
                   pd.DataFrame(list(table["cycles_per_class"]))], axis=1)
table

### Explore a recording
Pick a recording from the menu. The plot shows the waveform, with each annotated cycle coloured by its label.
Untick *Show normal cycles* to see only the abnormal ones. Use the player to listen.

**What to look and listen for:**
- **Crackles** are short, explosive, popping sounds (like hair rubbed between fingers).
- **Wheezes** are continuous, musical, whistling sounds.
- Compare devices: how different do the background noise and loudness sound?

In [ ]:
def show_recording(name, show_normal):
    audio, sr = load_recording(name)
    cycles = load_cycles(name)
    t = np.arange(len(audio)) / sr
    fig, ax = plt.subplots(figsize=(13, 3.2))
    ax.plot(t, audio, lw=0.5, color="#34495e")
    shade_cycles(ax, cycles, show_normal)
    ax.set(xlabel="Time (s)", ylabel="Amplitude", title=name, xlim=(0, t[-1]))
    plt.show()
    print("Cycles per class:", cycles.label.value_counts().to_dict())
    play(audio, sr)

W.interact(show_recording,
           name=W.Dropdown(options=REC_OPTIONS, description="Recording", layout=W.Layout(width="520px")),
           show_normal=W.Checkbox(value=True, description="Show normal cycles"));

> ✏️ **Try it yourself.** Load one recording with `load_recording(...)` and `load_cycles(...)` and find its longest cycle. Which class is it?

In [ ]:
# Your code here

## 3. Filtering

Lung sounds of interest sit roughly between **100 Hz and 2000 Hz**. Below that range we mostly find heart
sounds and muscle or movement noise; above it, mostly sensor and ambient noise.

In our pipeline every recording is first **resampled to 4 kHz** and then passed through a
**5th-order Butterworth band-pass filter (80 to 1900 Hz)**.

Use the controls to try different settings:
- **Sample rate**: the rate the recording is resampled to. A filter can only work below half the sample rate (the *Nyquist frequency*).
- **Filter**: none, low-pass (keeps low frequencies), high-pass (keeps high frequencies), or band-pass (keeps a range).
- **Low / high cutoff** and **order**: a higher order gives a sharper transition between kept and removed frequencies.

The left plot compares the waveforms, the right plot shows how much energy is at each frequency
before (grey) and after (blue) filtering. Listen to both versions below the plots.

In [ ]:
def show_filter(name, sr, kind, low, high, order):
    audio, sr = load_recording(name, sr=None if sr == 0 else sr)
    filtered = butterworth(audio, sr, kind, low, high, order)
    t = np.arange(len(audio)) / sr
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 3.6), gridspec_kw={"width_ratios": [1.6, 1]})
    a1.plot(t, audio, lw=0.5, color="#b0b0b0", label="original")
    a1.plot(t, filtered, lw=0.5, color="#2e86de", label="filtered")
    a1.set(xlabel="Time (s)", ylabel="Amplitude", title=f"Waveform at {sr} Hz", xlim=(0, t[-1]))
    a1.legend(loc="upper right")
    for x, color in [(audio, "#b0b0b0"), (filtered, "#2e86de")]:
        f, p = sps.welch(x, fs=sr, nperseg=2048)
        a2.semilogy(f, p + 1e-14, color=color)
    if kind in ("high-pass", "band-pass"): a2.axvline(low, color="k", ls="--", lw=0.8)
    if kind in ("low-pass", "band-pass"): a2.axvline(min(high, sr / 2), color="k", ls="--", lw=0.8)
    a2.set(xlabel="Frequency (Hz)", ylabel="Power", title="Power spectrum", xlim=(0, sr / 2))
    plt.tight_layout(); plt.show()
    if high >= sr / 2 and kind in ("low-pass", "band-pass"):
        print(f"Note: the high cutoff is above the Nyquist frequency ({sr / 2:.0f} Hz), so it was lowered to just below it.")
    print("Original:"); play(audio, sr)
    print("Filtered:"); play(filtered, sr)

W.interact(show_filter,
           name=W.Dropdown(options=REC_OPTIONS, description="Recording", layout=W.Layout(width="520px")),
           sr=W.Dropdown(options=[("original", 0), ("16000 Hz", 16000), ("8000 Hz", 8000), ("4000 Hz", 4000)],
                         value=4000, description="Sample rate"),
           kind=W.Dropdown(options=["none", "low-pass", "high-pass", "band-pass"], value="band-pass", description="Filter"),
           low=W.IntSlider(value=80, min=20, max=1000, step=10, description="Low cutoff (Hz)", style={"description_width": "initial"}),
           high=W.IntSlider(value=1900, min=200, max=8000, step=50, description="High cutoff (Hz)", style={"description_width": "initial"}),
           order=W.IntSlider(value=5, min=1, max=10, description="Order"));

> ✏️ **Try it yourself.** Apply `butterworth(...)` with a narrow band (for example 400 to 600 Hz) to a recording with wheezes. Can you still hear the wheeze?

In [ ]:
# Your code here

## 4. Segmentation

Deep learning models need inputs of a **fixed size**, but breathing cycles have different lengths.
We cut each annotated cycle out of the filtered recording and bring it to a fixed duration:
- cycles **shorter** than the target are **padded** at the end,
- cycles **longer** than the target are **truncated** (the end is cut off).

In our experiments the target was **4 s** for most models and **8 s** for OPERA.

Try different durations and padding methods:
- **reflect**: mirrors the end of the cycle (what we used),
- **zeros**: adds silence,
- **repeat**: repeats the cycle from the start.

The top plot shows the full recording with its cycles. The grid below shows each cycle after
padding or truncation; the shaded part is padding that was added.

In [ ]:
def show_segments(name, sr, duration, mode, show_normal):
    audio, sr = load_recording(name, sr=sr)
    audio = butterworth(audio, sr, "band-pass", 80, 1900, 5)
    cycles = load_cycles(name)
    if not show_normal:
        cycles = cycles[cycles.label != "Normal"]
    target = int(duration * sr)

    t = np.arange(len(audio)) / sr
    fig, ax = plt.subplots(figsize=(13, 2.6))
    ax.plot(t, audio, lw=0.5, color="#34495e")
    shade_cycles(ax, cycles, True)
    ax.set(xlabel="Time (s)", title=f"{name} (filtered, {sr} Hz)", xlim=(0, t[-1]))
    plt.show()

    n = len(cycles)
    if n == 0:
        print("No cycles to show."); return
    cols = 4; rows = int(np.ceil(n / cols))
    fig, axes = plt.subplots(rows, cols, figsize=(13, 2.0 * rows), squeeze=False)
    n_trunc = 0
    for ax, (_, c) in zip(axes.flat, cycles.iterrows()):
        seg = audio[int(c.start * sr):int(c.end * sr)]
        fixed = fix_length(seg, target, mode)
        n_trunc += len(seg) > target
        ts = np.arange(target) / sr
        ax.plot(ts, fixed, lw=0.5, color=CLASS_COLORS[c.label])
        if len(seg) < target:
            ax.axvspan(len(seg) / sr, duration, color="#f1c40f", alpha=0.2, lw=0)
        ax.set_title(f"{c.label}: {c.end - c.start:.2f} s", fontsize=9)
        ax.set_xlim(0, duration); ax.set_yticks([])
    for ax in list(axes.flat)[n:]:
        ax.axis("off")
    plt.tight_layout(); plt.show()
    print(f"{n} cycles, {n_trunc} truncated, {n - n_trunc} padded to {duration} s.")

W.interact(show_segments,
           name=W.Dropdown(options=REC_OPTIONS, description="Recording", layout=W.Layout(width="520px")),
           sr=W.Dropdown(options=[4000, 8000, 16000], value=4000, description="Sample rate"),
           duration=W.FloatSlider(value=4.0, min=1.0, max=10.0, step=0.5, description="Duration (s)"),
           mode=W.Dropdown(options=["reflect", "zeros", "repeat"], description="Padding"),
           show_normal=W.Checkbox(value=False, description="Show normal cycles"));

> ✏️ **Try it yourself.** Compute the length of every cycle in all 12 recordings (use `load_cycles`). What fraction would be truncated at 4 s? And at 8 s?

In [ ]:
# Your code here

## 5. Mel spectrogram

A **mel spectrogram** shows how the energy of a sound is distributed over time (x axis) and frequency
(y axis), with the frequency axis warped to match human hearing (the *mel* scale).

Vision models such as ResNet, EfficientNet and Swin were pretrained on photos (ImageNet), so we feed them
the mel spectrogram **as an image**: scaled to 0 to 255 grey levels and resized to 256 × 256 pixels.
Audio models (VGGish, YAMNet, OPERA) instead take the waveform and compute their own spectrogram internally.

Our settings: `n_fft = 512`, `hop = 128`, `n_mels = 128`, `fmax = 2000 Hz` at 4 kHz.

Try the controls:
- **n_fft**: window length. Longer windows give finer frequency detail but blur fast events like crackles.
- **hop**: step between windows. Smaller hops give more time frames.
- **n_mels**: number of frequency bands.

Crackles appear as short vertical lines; wheezes appear as horizontal lines (a steady tone).

In [ ]:
cycle_slider = W.IntSlider(value=0, min=0, description="Cycle #")
mel_rec = W.Dropdown(options=REC_OPTIONS, description="Recording", layout=W.Layout(width="520px"))

def _update_cycles(*_):
    cycle_slider.max = len(load_cycles(mel_rec.value)) - 1
mel_rec.observe(_update_cycles, "value"); _update_cycles()

def show_mel(name, cycle_idx, n_fft, hop, n_mels, fmax):
    sr = 4000
    audio, _ = load_recording(name, sr=sr)
    audio = butterworth(audio, sr, "band-pass", 80, 1900, 5)
    c = load_cycles(name).iloc[min(cycle_idx, len(load_cycles(name)) - 1)]
    seg = fix_length(audio[int(c.start * sr):int(c.end * sr)], 4 * sr)
    S = mel_db(seg, sr, n_fft, hop, n_mels, fmax)
    img = (S - S.min()) / (S.max() - S.min() + 1e-6)
    img256 = torch.nn.functional.interpolate(torch.tensor(img)[None, None], size=(256, 256),
                                             mode="bilinear", align_corners=False)[0, 0].numpy()
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(14, 3.8), gridspec_kw={"width_ratios": [1.7, 1]})
    im = librosa.display.specshow(S, sr=sr, hop_length=hop, x_axis="time", y_axis="mel", fmax=fmax, ax=a1, cmap="magma")
    fig.colorbar(im, ax=a1, format="%+.0f dB")
    a1.set_title(f"Cycle {cycle_idx}: {c.label} ({c.end - c.start:.2f} s, padded or cut to 4 s)  |  shape {S.shape}")
    a2.imshow(img256, cmap="gray", origin="lower", aspect="auto")
    a2.set(title="What a vision model sees (256 x 256, grey)", xticks=[], yticks=[])
    plt.tight_layout(); plt.show()
    play(seg, sr)

import librosa.display
W.interact(show_mel, name=mel_rec, cycle_idx=cycle_slider,
           n_fft=W.Dropdown(options=[128, 256, 512, 1024], value=512, description="n_fft"),
           hop=W.Dropdown(options=[32, 64, 128, 256], value=128, description="hop"),
           n_mels=W.Dropdown(options=[32, 64, 128], value=128, description="n_mels"),
           fmax=W.IntSlider(value=2000, min=500, max=2000, step=100, description="fmax (Hz)"));

> ✏️ **Try it yourself.** Compute the mel spectrogram of one cycle with `mel_db(...)` and compare its shape for `hop=64` and `hop=256`. Why does it change?

In [ ]:
# Your code here

## 6. Train a classifier on frozen features

Training a large network from scratch needs far more data than ICBHI has. Instead we use **pretrained
models as frozen feature extractors**: each cycle goes through the model once, and we keep its output
vector (the *embedding*). We already did this for the whole ICBHI dataset, so you don't need a GPU.

On top of these embeddings we train only a small **classification head**: a single linear layer that maps
the embedding to the 4 classes. This takes a few seconds on a laptop or in Colab.

The six models:

| Model | Pretrained on | Input |
|---|---|---|
| ResNet-50 | ImageNet (photos) | mel spectrogram image |
| EfficientNetV2-S | ImageNet (photos) | mel spectrogram image |
| Swin Transformer V2-B | ImageNet (photos) | mel spectrogram image |
| VGGish | AudioSet (YouTube audio) | waveform |
| YAMNet | AudioSet (YouTube audio) | waveform |
| OPERA-CT | Respiratory sounds (breathing, cough, lung) | waveform |

**Choose your settings and press Train:**
- **Model**: which embeddings to use.
- **Optimizer**: Adam, AdamW or SGD, and its learning rate.
- **Class weights**: ICBHI is imbalanced (about half the cycles are Normal). With class weights, mistakes on rare classes cost more, which usually raises sensitivity.

We train for a fixed number of epochs and keep the epoch with the best validation ICBHI score.

In [ ]:
EPOCHS, BATCH = 30, 32
torch.set_num_threads(1)  # the head is tiny: one thread is fastest and avoids thread contention
MODEL_NAMES = {"resnet50": "ResNet-50", "efficientnet_v2_s": "EfficientNetV2-S", "swin_v2_b": "Swin-V2-B",
               "vggish": "VGGish", "yamnet": "YAMNet", "opera_ct": "OPERA-CT"}
RESULT = {}


def icbhi_scores(y_true, y_pred):
    """Official ICBHI metrics. Se: abnormal cycles classified exactly right. Sp: normal cycles classified as normal."""
    normal = CLASSES.index("Normal")
    abn = y_true != normal
    se = np.mean(y_pred[abn] == y_true[abn]); sp = np.mean(y_pred[~abn] == normal)
    return {"Sensitivity": se, "Specificity": sp, "ICBHI score": (se + sp) / 2}


def load_split(model, split):
    X = np.load(DATA / "embeddings" / f"{model}_{split}.npy").astype(np.float32)
    lbl = MANIFEST["labels"]["train"][model] if split == "train" else MANIFEST["labels"][split]
    return X, np.load(DATA / lbl).astype(np.int64)


def load_model_data(model):
    Xtr, ytr = load_split(model, "train")
    Xte, yte = load_split(model, "test")
    if (DATA / "embeddings" / f"{model}_val.npy").exists():
        Xva, yva = load_split(model, "val")
    else:  # OPERA: no saved validation split, so hold out 15% of train (fixed seed)
        idx = np.random.default_rng(0).permutation(len(ytr)); n = int(0.15 * len(ytr))
        Xva, yva, Xtr, ytr = Xtr[idx[:n]], ytr[idx[:n]], Xtr[idx[n:]], ytr[idx[n:]]
    return Xtr, ytr, Xva, yva, Xte, yte


def train_head(model, optimizer="Adam", lr=1e-3, class_weights=True, verbose=True):
    torch.manual_seed(0)
    Xtr, ytr, Xva, yva, Xte, yte = load_model_data(model)
    mu, sd = Xtr.mean(0), Xtr.std(0) + 1e-6            # standardise with training statistics
    norm = lambda X: torch.tensor((X - mu) / sd)
    head = nn.Linear(Xtr.shape[1], len(CLASSES))
    w = None
    if class_weights:
        counts = np.bincount(ytr, minlength=len(CLASSES))
        w = torch.tensor(len(ytr) / (len(CLASSES) * counts), dtype=torch.float32)
    loss_fn = nn.CrossEntropyLoss(weight=w)
    opt = {"Adam": torch.optim.Adam, "AdamW": torch.optim.AdamW,
           "SGD": lambda p, lr: torch.optim.SGD(p, lr=lr, momentum=0.9)}[optimizer](head.parameters(), lr=lr)
    Xt, yt, Xv = norm(Xtr), torch.tensor(ytr), norm(Xva)
    history, best = [], (-1, None, 0)
    for epoch in range(EPOCHS):
        head.train()
        perm = torch.randperm(len(yt)); total = 0.0
        for i in range(0, len(yt), BATCH):
            b = perm[i:i + BATCH]
            opt.zero_grad(); loss = loss_fn(head(Xt[b]), yt[b]); loss.backward(); opt.step()
            total += loss.item() * len(b)
        head.eval()
        with torch.no_grad():
            pv = head(Xv).argmax(1).numpy()
            val_loss = loss_fn(head(Xv), torch.tensor(yva)).item()
        s = icbhi_scores(yva, pv)["ICBHI score"]
        history.append({"epoch": epoch + 1, "train loss": total / len(yt), "val loss": val_loss, "val ICBHI score": s})
        if s > best[0]:
            best = (s, {k: v.clone() for k, v in head.state_dict().items()}, epoch + 1)
    head.load_state_dict(best[1]); head.eval()
    with torch.no_grad():
        prob = torch.softmax(head(norm(Xte)), 1).numpy()
    RESULT.clear()
    RESULT.update(model=model, head=head, mu=mu, sd=sd, history=pd.DataFrame(history), best_epoch=best[2],
                  y_test=yte, prob_test=prob, y_pred=prob.argmax(1),
                  settings={"optimizer": optimizer, "lr": lr, "class_weights": class_weights, "epochs": EPOCHS})
    if verbose:
        print(f"{MODEL_NAMES[model]}: {Xtr.shape[1]}-d embeddings, {len(ytr)} train / {len(yva)} val / {len(yte)} test cycles")
        print(f"Best validation ICBHI score {best[0]:.3f} at epoch {best[2]} of {EPOCHS}")
    return RESULT


def plot_history():
    h = RESULT["history"]
    fig, (a1, a2) = plt.subplots(1, 2, figsize=(12, 3.2))
    a1.plot(h.epoch, h["train loss"], label="train"); a1.plot(h.epoch, h["val loss"], label="validation")
    a1.set(xlabel="Epoch", ylabel="Loss", title="Loss"); a1.legend()
    a2.plot(h.epoch, h["val ICBHI score"], color="#27ae60"); a2.axvline(RESULT["best_epoch"], ls="--", color="k", lw=0.8)
    a2.set(xlabel="Epoch", ylabel="Score", title="Validation ICBHI score (dashed: kept epoch)")
    plt.tight_layout(); plt.show()


w_model = W.Dropdown(options=[(MODEL_NAMES[m], m) for m in MANIFEST["models"]], description="Model")
w_opt = W.Dropdown(options=["Adam", "AdamW", "SGD"], description="Optimizer")
w_lr = W.Dropdown(options=[1e-2, 1e-3, 1e-4], value=1e-3, description="Learning rate", style={"description_width": "initial"})
w_cw = W.Checkbox(value=True, description="Use class weights")
w_btn = W.Button(description="Train", button_style="primary", icon="play")
w_out = W.Output()

def _on_train(_):
    with w_out:
        clear_output(wait=True)
        info = MANIFEST["models"][w_model.value]
        if "NOT" in info.get("caveat", ""):
            print("Note:", info["caveat"].split(".")[0] + ".")
        print("Training...")
        train_head(w_model.value, w_opt.value, w_lr.value, w_cw.value)
        plot_history()
        print("Done. Run the next section to see the results.")
w_btn.on_click(_on_train)
display(W.VBox([W.HBox([w_model, w_opt]), W.HBox([w_lr, w_cw, w_btn]), w_out]))

> 💡 **Ideas to try:** train the same model with and without class weights, then compare sensitivity in the
> next section. Then compare an ImageNet model with an audio model. Which features work better for lung sounds?

## 7. Results

We evaluate on the **official ICBHI test set**, which contains patients the model never saw during training.

The ICBHI challenge uses three metrics:
- **Sensitivity (Se)**: of all *abnormal* cycles, how many got exactly the right class.
- **Specificity (Sp)**: of all *normal* cycles, how many were recognised as normal.
- **ICBHI score**: the average of Se and Sp. It is used because plain accuracy looks good even for a model that calls everything Normal.

Pick which metrics to show. Re-run this cell after training another model.

In [ ]:
def show_results(metrics):
    if not RESULT:
        print("Train a model in section 6 first."); return
    y, p = RESULT["y_test"], RESULT["y_pred"]
    print(f"Model: {MODEL_NAMES[RESULT['model']]}   settings: {RESULT['settings']}")
    values = dict(icbhi_scores(y, p))
    values["Accuracy"] = accuracy_score(y, p)
    values["Macro F1"] = f1_score(y, p, average="macro")
    rows = {k: f"{v * 100:.1f} %" for k, v in values.items() if k in metrics}
    if rows:
        display(pd.DataFrame(rows, index=["test"]))
    if "Per-class recall" in metrics:
        rec = {c: np.mean(p[y == i] == i) for i, c in enumerate(CLASSES)}
        display(pd.DataFrame({k: f"{v * 100:.1f} %" for k, v in rec.items()}, index=["recall"]))
    if "Confusion matrix" in metrics:
        cm = confusion_matrix(y, p, labels=range(len(CLASSES)))
        fig, ax = plt.subplots(figsize=(4.6, 4))
        ax.imshow(cm, cmap="Blues"); ax.grid(False)
        for i in range(len(CLASSES)):
            for j in range(len(CLASSES)):
                ax.text(j, i, cm[i, j], ha="center", va="center", color="white" if cm[i, j] > cm.max() / 2 else "black")
        ax.set(xticks=range(len(CLASSES)), yticks=range(len(CLASSES)), xticklabels=CLASSES, yticklabels=CLASSES,
               xlabel="Predicted", ylabel="True", title="Confusion matrix (test)")
        plt.show()

W.interact(show_results, metrics=W.SelectMultiple(
    options=["Sensitivity", "Specificity", "ICBHI score", "Accuracy", "Macro F1", "Per-class recall", "Confusion matrix"],
    value=["Sensitivity", "Specificity", "ICBHI score", "Confusion matrix"], rows=7, description="Metrics"));

### Look at individual predictions
Each row is one test cycle with its true label, the predicted label, and how confident the model was.
Tick *Only mistakes* to see where the model goes wrong.

In [ ]:
def show_predictions(n, only_mistakes, seed):
    if not RESULT:
        print("Train a model in section 6 first."); return
    y, p, prob = RESULT["y_test"], RESULT["y_pred"], RESULT["prob_test"]
    idx = np.where(p != y)[0] if only_mistakes else np.arange(len(y))
    idx = np.random.default_rng(seed).choice(idx, size=min(n, len(idx)), replace=False)
    df = pd.DataFrame({"test cycle": idx, "true": [CLASSES[i] for i in y[idx]], "predicted": [CLASSES[i] for i in p[idx]],
                       "confidence": [f"{prob[i, p[i]] * 100:.0f} %" for i in idx]})
    df["correct"] = np.where(df["true"] == df["predicted"], "✓", "✗")
    display(df.reset_index(drop=True))

W.interact(show_predictions, n=W.IntSlider(value=10, min=5, max=30, description="Rows"),
           only_mistakes=W.Checkbox(value=False, description="Only mistakes"),
           seed=W.IntSlider(value=0, min=0, max=20, description="Shuffle"));

> ✏️ **Try it yourself.** Using `RESULT['y_test']` and `RESULT['y_pred']`, which class is most often confused with Normal? Why might that be?

In [ ]:
# Your code here

## 8. Export your classification head

Save the head you just trained as an **ONNX** file, a standard format that runs in many tools and in the
browser. You can upload it to the workshop website to try it interactively.

The file contains only the small head (with the standardisation built in), not the large pretrained model.
Its input is one embedding from the model you chose, and its output is the probability of each of the four classes.
A small `.json` file with the same name describes the input, the class order and your test scores.

In [ ]:
def export_head(path_stem=None):
    if not RESULT:
        print("Train a model in section 6 first."); return
    model = RESULT["model"]
    stem = path_stem or f"head_{model}"
    lin = RESULT["head"]
    # Fold the standardisation (x - mu) / sd into the linear layer so the ONNX file takes raw embeddings.
    mu, sd = torch.tensor(RESULT["mu"]), torch.tensor(RESULT["sd"])
    folded = nn.Linear(lin.in_features, lin.out_features)
    with torch.no_grad():
        folded.weight.copy_(lin.weight / sd)
        folded.bias.copy_(lin.bias - (lin.weight * (mu / sd)).sum(1))
    exported = nn.Sequential(folded, nn.Softmax(dim=1)).eval()
    dummy = torch.zeros(1, lin.in_features)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        torch.onnx.export(exported, dummy, f"{stem}.onnx", input_names=["embedding"], output_names=["probabilities"],
                          dynamic_axes={"embedding": {0: "batch"}, "probabilities": {0: "batch"}}, opset_version=17,
                          dynamo=False)
    scores = icbhi_scores(RESULT["y_test"], RESULT["y_pred"])
    meta = {"backbone": model, "backbone_name": MODEL_NAMES[model], "embedding_dim": lin.in_features,
            "input": "embedding", "output": "probabilities", "classes": CLASSES,
            "training": RESULT["settings"], "test_scores": {k: round(float(v), 4) for k, v in scores.items()}}
    Path(f"{stem}.json").write_text(json.dumps(meta, indent=2))

    # Check: the ONNX file gives the same answers as the trained head
    import onnxruntime as ort
    Xte, _ = load_split(model, "test")
    onnx_prob = ort.InferenceSession(f"{stem}.onnx").run(None, {"embedding": Xte[:200]})[0]
    agree = np.mean(onnx_prob.argmax(1) == RESULT["y_pred"][:200])
    print(f"Saved {stem}.onnx and {stem}.json  (ONNX matches the trained head on {agree * 100:.0f}% of 200 test cycles)")
    try:
        from google.colab import files
        files.download(f"{stem}.onnx"); files.download(f"{stem}.json")
    except ImportError:
        print(f"Files are in: {Path.cwd()}")

w_exp = W.Button(description="Export ONNX", button_style="success", icon="download")
w_exp_out = W.Output()
def _on_export(_):
    with w_exp_out:
        clear_output(wait=True); export_head()
w_exp.on_click(_on_export)
display(W.VBox([w_exp, w_exp_out]))

## 9. Wrap-up

What you saw today:
- Recordings from different devices sound and look very different. This **device variability** is one reason models trained on ICBHI struggle to generalise.
- Preprocessing choices (filter band, segment length, padding) change what the model receives.
- Frozen pretrained models give useful features without a GPU, but **sensitivity stays low**: abnormal sounds are hard to separate, and the class imbalance pushes models towards predicting Normal.

**Questions to think about:**
- Would fine-tuning the whole network help, or would it overfit on such a small dataset?
- How could we make a model robust to a new stethoscope it has never seen?
- Is the ICBHI score the right metric for a screening tool? What would a clinician care about most?

**Reference:** B. M. Rocha et al., "An open access database for the evaluation of respiratory sound
classification algorithms," *Physiological Measurement*, 40(3), 035001, 2019.